# Regression and Classification Workflows

Train one regression model and one classification model using scikit-learn.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 500
df = pd.DataFrame({
    "size_sqft": rng.normal(1600, 450, n).clip(500, 3500),
    "bedrooms": rng.integers(1, 6, n),
    "age_years": rng.integers(0, 80, n),
})
noise = rng.normal(0, 25000, n)
df["price"] = 85000 + 210*df["size_sqft"] + 18000*df["bedrooms"] - 1200*df["age_years"] + noise
df.head()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="price"), df["price"], test_size=0.2, random_state=RANDOM_STATE
)
reg_model = LinearRegression().fit(X_train, y_train)
reg_pred = reg_model.predict(X_test)
regression_metrics = {
    "mae": mean_absolute_error(y_test, reg_pred),
    "r2": r2_score(y_test, reg_pred)
}
regression_metrics


In [ ]:
data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=RANDOM_STATE
)
clf_model = Pipeline([
    ("scale", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=3000))
])
clf_model.fit(X_train, y_train)
clf_pred = clf_model.predict(X_test)
classification_metrics = {
    "accuracy": accuracy_score(y_test, clf_pred),
    "report": classification_report(y_test, clf_pred, output_dict=True)
}
classification_metrics["accuracy"]


In [ ]:
with open("metrics.json", "w") as f:
    json.dump({"regression": regression_metrics, "classification": classification_metrics}, f, indent=2)
